## Example 1: Basic Discrete Wavelet Transform

In [ ]:
# Example 1: Basic Discrete Wavelet Transform
import pywt  # PyWavelets is imported as 'pywt'
import numpy as np

print("=== Basic Wavelet Transform ===")

print(f"PyWavelets version: {pywt.__version__}")

# Create test signal
test_signal = np.array([1, 4, 5, 2, 8, 3, 6, 7, 1, 9])
print(f"Original signal: {test_signal}")

# Available wavelets
wavelets = pywt.wavelist()
print(f"Available wavelets: {len(wavelets)} total")
print(f"Sample wavelets: {wavelets[:8]}")

# Basic DWT with different wavelets
wavelets_to_test = ['haar', 'db4', 'db8', 'sym5']

for wavelet in wavelets_to_test:
    try:
        # Discrete Wavelet Transform
        coeffs = pywt.dwt(test_signal, wavelet)
        cA, cD = coeffs  # Approximation and Detail coefficients
        
        print(f"\nWavelet: {wavelet}")
        print(f"  Approximation (cA): {cA[:4]}... (shape: {cA.shape})")
        print(f"  Detail (cD): {cD[:4]}... (shape: {cD.shape})")
        
        # Inverse DWT to reconstruct signal
        reconstructed = pywt.idwt(cA, cD, wavelet)
        reconstruction_error = np.mean(np.abs(test_signal - reconstructed[:len(test_signal)]))
        print(f"  Reconstruction error: {reconstruction_error:.6f}")
        # Single-level DWT gives floor((N + filter_length - 1) / 2) coefficients in each half
        expected_len = pywt.dwt_coeff_len(len(test_signal), pywt.Wavelet(wavelet).dec_len, 'symmetric')
        assert cA.shape == cD.shape == (expected_len,)
        assert np.allclose(reconstructed[:len(test_signal)], test_signal)
        
    except Exception as e:
        print(f"Error with {wavelet}: {e}")
        raise

print("Basic wavelet transform completed!")
assert set(wavelets_to_test) <= set(wavelets)
# Haar approximation coefficients are pairwise sums divided by sqrt(2)
cA_haar, cD_haar = pywt.dwt(test_signal, 'haar')
assert np.allclose(cA_haar, (test_signal[0::2] + test_signal[1::2]) / np.sqrt(2))
assert np.allclose(cD_haar, (test_signal[0::2] - test_signal[1::2]) / np.sqrt(2))

## Example 2: Multilevel Wavelet Decomposition

In [ ]:
# Example 2: Multilevel Wavelet Decomposition
import pywt
import numpy as np

print("=== Multilevel Wavelet Decomposition ===")

# Create a more complex signal with noise
np.random.seed(42)
t = np.linspace(0, 2*np.pi, 128)
clean_signal = np.sin(2*t) + 0.5*np.sin(8*t) + 0.3*np.sin(16*t)
noisy_signal = clean_signal + 0.2*np.random.randn(len(t))

print(f"Signal length: {len(noisy_signal)}")
print(f"Signal range: [{noisy_signal.min():.3f}, {noisy_signal.max():.3f}]")

# Multilevel decomposition 
wavelet = 'db6'
levels = 4

print(f"\nMultilevel decomposition with {wavelet}, {levels} levels:")

# Wavelet decomposition
coeffs = pywt.wavedec(noisy_signal, wavelet, level=levels)
print(f"Number of coefficient arrays: {len(coeffs)}")

for i, coeff in enumerate(coeffs):
    if i == 0:
        print(f"  Level {i} (approximation): {coeff.shape} - range [{coeff.min():.3f}, {coeff.max():.3f}]")
    else:
        print(f"  Level {i} (detail): {coeff.shape} - range [{coeff.min():.3f}, {coeff.max():.3f}]")

# Reconstruct signal
reconstructed = pywt.waverec(coeffs, wavelet)
reconstruction_error = np.mean(np.abs(noisy_signal - reconstructed))
print(f"\nReconstruction error: {reconstruction_error:.6f}")

# Denoising by thresholding detail coefficients
threshold = 0.1
coeffs_thresh = list(coeffs)
for i in range(1, len(coeffs_thresh)):
    coeffs_thresh[i] = pywt.threshold(coeffs_thresh[i], threshold, mode='soft')

denoised_signal = pywt.waverec(coeffs_thresh, wavelet)
denoising_improvement = np.mean(np.abs(clean_signal - noisy_signal)) - np.mean(np.abs(clean_signal - denoised_signal[:len(clean_signal)]))
print(f"Denoising improvement: {denoising_improvement:.6f}")

print("Multilevel decomposition completed!")
assert len(coeffs) == levels + 1
assert [len(c) for c in coeffs] == [18, 18, 25, 40, 69]
assert np.allclose(reconstructed, noisy_signal)
assert denoising_improvement > 0   # thresholding moved the signal closer to the clean one

## Example 3: 2D Wavelet Transform for Image Processing

In [ ]:
# Example 3: 2D Wavelet Transform for Image Processing
import pywt
import numpy as np

print("=== 2D Wavelet Transform ===")

# Create a synthetic 2D signal (simulating an image)
np.random.seed(42)
size = 64
x, y = np.meshgrid(np.linspace(-2, 2, size), np.linspace(-2, 2, size))

# Create patterns: circles + sine waves + noise
image = (np.exp(-(x**2 + y**2)/0.5) + 
         0.3 * np.sin(4*np.pi*x) * np.sin(4*np.pi*y) + 
         0.1 * np.random.randn(size, size))

print(f"Image shape: {image.shape}")
print(f"Image range: [{image.min():.3f}, {image.max():.3f}]")

# 2D DWT
wavelet = 'db4'
coeffs2d = pywt.dwt2(image, wavelet)

# Extract coefficient arrays
cA, (cH, cV, cD) = coeffs2d
print(f"\n2D DWT coefficients:")
print(f"  Approximation (cA): {cA.shape}")
print(f"  Horizontal detail (cH): {cH.shape}")
print(f"  Vertical detail (cV): {cV.shape}")
print(f"  Diagonal detail (cD): {cD.shape}")

# Coefficient statistics
for name, coeff in [('Approximation', cA), ('Horizontal', cH), ('Vertical', cV), ('Diagonal', cD)]:
    print(f"  {name}: mean={coeff.mean():.3f}, std={coeff.std():.3f}, max_abs={np.abs(coeff).max():.3f}")

# Inverse 2D DWT
reconstructed = pywt.idwt2(coeffs2d, wavelet)
reconstruction_error = np.mean(np.abs(image - reconstructed))
print(f"\n2D reconstruction error: {reconstruction_error:.6f}")

# Multilevel 2D decomposition
levels = 3
coeffs_ml = pywt.wavedec2(image, wavelet, level=levels)
print(f"\nMultilevel 2D decomposition ({levels} levels):")
print(f"  Number of levels: {len(coeffs_ml)}")
print(f"  Approximation shape: {coeffs_ml[0].shape}")

for i, level_coeffs in enumerate(coeffs_ml[1:], 1):
    cH, cV, cD = level_coeffs
    print(f"  Level {i} details: H={cH.shape}, V={cV.shape}, D={cD.shape}")

print("2D wavelet transform completed!")
assert cA.shape == cH.shape == cV.shape == cD.shape
assert pywt.dwt2(image, wavelet)[0].shape == (35, 35)
assert np.allclose(reconstructed, image)
assert len(coeffs_ml) == levels + 1 and coeffs_ml[0].shape == (14, 14)
assert [level[0].shape for level in coeffs_ml[1:]] == [(14, 14), (21, 21), (35, 35)]
assert np.allclose(pywt.waverec2(coeffs_ml, wavelet), image)

## Example 4: Wavelet Families and Properties

In [ ]:
# Example 4: Wavelet Families and Properties
import pywt
import numpy as np

print("=== Wavelet Families and Properties ===")

# Explore wavelet families
families = pywt.families()
print(f"Wavelet families: {families}")

# Detail each family
for family in families[:6]:  # First 6 families
    wavelets_in_family = pywt.wavelist(family)
    print(f"\n{family.upper()} family: {len(wavelets_in_family)} wavelets")
    print(f"  Examples: {wavelets_in_family[:5]}")

# Wavelet properties
print(f"\n=== Wavelet Properties ===")
wavelets_to_analyze = ['haar', 'db4', 'db8', 'sym8', 'coif4', 'bior2.2']

for wavelet_name in wavelets_to_analyze:
    # Create wavelet object
    wavelet = pywt.Wavelet(wavelet_name)
    
    print(f"\nWavelet: {wavelet_name}")
    print(f"  Family: {wavelet.family_name}")
    print(f"  Short name: {wavelet.short_family_name}")  # Wavelet.short_family_name, e.g. 'db' for Daubechies
    print(f"  Orthogonal: {wavelet.orthogonal}")
    print(f"  Biorthogonal: {wavelet.biorthogonal}")
    print(f"  Symmetry: {wavelet.symmetry}")
    print(f"  DWT length: {wavelet.dec_len}")
    print(f"  Reconstruction length: {wavelet.rec_len}")
    
    # Wavelet and scaling function values: an orthogonal wavelet returns (phi, psi, x); a
    # biorthogonal one returns decomposition AND reconstruction functions (phi_d, psi_d, phi_r, psi_r, x).
    if wavelet.orthogonal:
        phi, psi, x = wavelet.wavefun(level=5)
    else:
        phi, psi, phi_r, psi_r, x = wavelet.wavefun(level=5)
        assert len(phi_r) == len(psi_r) == len(x)
    assert len(phi) == len(psi) == len(x) > 0
    print(f"  Scaling function samples: {len(phi)}")
    print(f"  Wavelet function samples: {len(psi)}")

# Filter bank properties
print(f"\n=== Filter Bank Properties ===")
test_wavelets = ['db2', 'haar', 'bior1.1']

for wname in test_wavelets:
    wavelet = pywt.Wavelet(wname)
    print(f"\n{wname} filters:")
    print(f"  Decomposition low-pass: {len(wavelet.dec_lo)} coeffs")
    print(f"  Decomposition high-pass: {len(wavelet.dec_hi)} coeffs")
    print(f"  Reconstruction low-pass: {len(wavelet.rec_lo)} coeffs")
    print(f"  Reconstruction high-pass: {len(wavelet.rec_hi)} coeffs")

print("Wavelet family analysis completed!")

## Example 5: Wavelet Packet Analysis

In [ ]:
import pywt
import numpy as np

# Example 5: Wavelet Packet Analysis
print("Testing pywavelets wavelet packets:")

# Create test signal
t = np.linspace(0, 4*np.pi, 128)
signal = 2*np.sin(2*t) + np.sin(8*t) + 0.5*np.sin(20*t)
print(f"✓ Signal length: {len(signal)}")

# Wavelet packet decomposition: unlike the DWT, both the approximation (a) and the detail (d)
# branches are split again at every level
wp = pywt.WaveletPacket(data=signal, wavelet='db4', mode='symmetric')
print(f"✓ Wavelet packet created, max level: {wp.maxlevel}")
assert wp.maxlevel == 4

# Analyze different packet nodes
nodes_to_check = ['a', 'd', 'aa', 'ad', 'da', 'dd']
for node_name in nodes_to_check:
    node = wp[node_name]
    energy = np.sum(node.data**2)
    print(f"✓ Node '{node_name}': {len(node.data)} coeffs, energy={energy:.2f}")
assert [len(wp[n].data) for n in nodes_to_check] == [67, 67, 37, 37, 37, 37]
assert [node.path for node in wp.get_level(2, order='natural')] == ['aa', 'ad', 'da', 'dd']

# Reconstruction from the leaves gives the signal back
assert np.allclose(wp.reconstruct(update=False), signal)

# With periodization the transform is orthogonal, so the level-2 nodes split the signal energy exactly
wp_periodic = pywt.WaveletPacket(data=signal, wavelet='db4', mode='periodization')
energies = {node.path: np.sum(node.data**2) for node in wp_periodic.get_level(2)}
print(f"✓ Level-2 energies (periodization): { {k: round(float(v), 2) for k, v in energies.items()} }")
assert np.isclose(sum(energies.values()), np.sum(signal**2))
assert max(energies, key=energies.get) == 'aa'   # most energy is in the lowest band